In [14]:
import requests
import json
import os
import urllib3
import boto3
import io
import gzip
import logging
import pandas as pd
from datetime import datetime

In [10]:
s3_client = boto3.client('s3')
today_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
bucket_name = "uncc_traffic_data"

In [11]:
s3_bucket = "s3-3601-final-project"
s3_key = f"tomtom/traffic/{today_str}speeds.json.gz"

In [12]:
api_key = 'AskBella'
#timestamp = datetime.utcnow()


url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"

data_list = []

latitude = [35.307778, 35.313611, 35.311361, 35.309306, 35.306917, 35.293111, 35.294889, 35.299639, 35.287972, 35.301861, 35.311861] # fill in with the found values
longitude = [-80.724583, -80.701944, -80.711222, -80.718056, -80.722639, -80.749361, -80.742111, -80.74575, -80.759472, -80.752333, -80.744]

for i in range(len(latitude)):

        params = {
            'key': api_key,
            'point': f"{latitude[i]},{longitude[i]}",
            'unit': "MPH",
            "openLr": "false"
        }

        response = requests.get(url, params)

        if response.status_code==200:
            data = response.json()
            flow_data = data.get("flowSegmentData", {})

            data_we_want = {
                "timestamp": datetime.now().strftime("%Y-%m-%d_%H-%M-%S"),
                "latitude": latitude[i],
                "longitude": longitude[i],
                "current_speed": flow_data.get("currentSpeed"),
                "free_flow_speed": flow_data.get("freeFlowSpeed"),
                "current_delay": flow_data.get("currentDelay"),
                "confidence": flow_data.get("confidence") # for flags of when we shoudl somehow retrain?
            }

            data_list.append(data_we_want)
        else:
            print(f"Error in retrieving data: {response.status_code}")

Error in retrieving data: 401
Error in retrieving data: 401
Error in retrieving data: 401
Error in retrieving data: 401
Error in retrieving data: 401
Error in retrieving data: 401
Error in retrieving data: 401
Error in retrieving data: 401
Error in retrieving data: 401
Error in retrieving data: 401
Error in retrieving data: 401


In [15]:
#making data_list into a DataFrame
df = pd.DataFrame(data_list) 
print(df.head())

Empty DataFrame
Columns: []
Index: []


In [16]:
#converting into a csv to save into our shared s3 bucket
df.to_csv("TomTom_traffic.csv", index=False)


s3_client.upload_file(
    "TomTom_traffic.csv",
    "s3-3601-final-project",
    "tomtom/traffic/speeds.json.gz"
)

In [17]:
compressed = io.BytesIO()
with gzip.GzipFile(fileobj=compressed, mode="wb") as gz:
    gz.write(json.dumps(data_list).encode("utf-8"))
                                    
s3_client.put_object(
    Bucket=s3_bucket,
    Key=s3_key,
    Body=compressed.getvalue(),
    ContentType="application/json",
    ContentEncoding="gzip",
)

{'ResponseMetadata': {'RequestId': 'KJRQA372RQXWNZZ7',
  'HostId': 'D4Hva9RiHkOMywF+jlrEBsDZDC9ViaT9oZB7dax8vxawSENxu7j4cnaeiNtaeUfWt2zBi+7IS951NWF3EiKduy7+dCPU1oR4',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'D4Hva9RiHkOMywF+jlrEBsDZDC9ViaT9oZB7dax8vxawSENxu7j4cnaeiNtaeUfWt2zBi+7IS951NWF3EiKduy7+dCPU1oR4',
   'x-amz-request-id': 'KJRQA372RQXWNZZ7',
   'date': 'Fri, 25 Sep 2026 19:46:46 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"000c92207403de065e8c0a9cc67e9c0a"',
   'x-amz-checksum-crc32': 'ITmKzg==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"000c92207403de065e8c0a9cc67e9c0a"',
 'ChecksumCRC32': 'ITmKzg==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}